# Production-oriented Bank Marketing Classification

A reproducible, leakage-aware workflow to predict whether a client subscribes (`y`). The notebook compares common tabular classifiers, tunes the best candidate, evaluates it on a held-out test set, and saves artifacts.

**Assumption:** predictions are made before a marketing call. Therefore, `duration` is excluded because it is only known after the call and would leak future information. If predicting after a call, include it deliberately.

No algorithm is universally best. Validate model quality and decision thresholds against the costs of missed subscriptions and unnecessary calls before deployment.

## 1. Imports and configuration

In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import randint, loguniform, uniform
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, classification_report, confusion_matrix,
    f1_score, log_loss, precision_score, recall_score,
    roc_auc_score, roc_curve, precision_recall_curve,
)
from sklearn.model_selection import (
    RandomizedSearchCV, StratifiedKFold, cross_validate,
    cross_val_predict, train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
DATA_PATH = Path("bank.csv")
TARGET = "y"
EXCLUDE_PRE_CALL = ["duration"]
REPORT_DIR = Path("model_reports")
REPORT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 100)

## 2. Load and validate the source data

The Bank Marketing CSV commonly uses semicolons. Keep the original file unchanged.

In [ ]:
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Could not find {DATA_PATH.resolve()}")

df = pd.read_csv(DATA_PATH, sep=";")
df.columns = df.columns.astype(str).str.strip()

if TARGET not in df.columns:
    raise ValueError(f"Expected target column {TARGET!r}; found {df.columns.tolist()}")
if df.empty:
    raise ValueError("The input CSV has no rows.")

target_text = df[TARGET].astype("string").str.strip().str.lower()
y = target_text.map({"no": 0, "yes": 1})
if y.isna().any():
    unexpected = target_text[y.isna()].value_counts(dropna=False).to_dict()
    raise ValueError(f"Target has missing or unexpected labels: {unexpected}")
if y.nunique() != 2:
    raise ValueError("Expected a binary target with both 'no' and 'yes' classes.")

print(f"Rows: {len(df):,}; columns: {df.shape[1]:,}")
display(df.head())

## 3. Data-quality and target checks

Duplicates and missing values are reported for investigation; they are not removed automatically.

In [ ]:
display(df.dtypes.rename("dtype").to_frame())

missing_report = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
})
display(missing_report.sort_values("missing_count", ascending=False))
print(f"Exact duplicate rows: {df.duplicated().sum():,}")
display(y.value_counts().rename(index={0: "no", 1: "yes"}).to_frame("count"))
display((y.value_counts(normalize=True) * 100).rename(index={0: "no", 1: "yes"}).round(2).to_frame("percent"))

numeric_columns = df.drop(columns=[TARGET]).select_dtypes(include="number").columns.tolist()
categorical_columns = df.drop(columns=[TARGET]).select_dtypes(exclude="number").columns.tolist()
print("Numeric columns:", numeric_columns)
print("Categorical columns:", categorical_columns)

## 4. Define features and hold out the test set

Use a stratified split so the class proportions remain similar. The test set is not used for model selection, tuning, or threshold selection.

In [ ]:
excluded_columns = [column for column in EXCLUDE_PRE_CALL if column in df.columns]
X = df.drop(columns=[TARGET, *excluded_columns]).copy()
print("Excluded from pre-call prediction:", excluded_columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {X_train.shape}; test: {X_test.shape}")
print("Train positive rate:", round(y_train.mean(), 4))
print("Test positive rate:", round(y_test.mean(), 4))

## 5. Leakage-safe preprocessing

Imputation, scaling, and category encoding are fitted inside each model pipeline and cross-validation fold. Dense one-hot output is used here for broad estimator compatibility; for very high-cardinality data, choose a sparse-compatible pipeline or native categorical model.

In [ ]:
numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(exclude="number").columns.tolist()

try:
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:  # scikit-learn versions before sparse_output
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_features),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", one_hot),
        ]), categorical_features),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

print(f"Numeric features: {len(numeric_features)}; categorical features: {len(categorical_features)}")

## 6. Candidate models

Compare a linear baseline, bagged trees, and boosting models. XGBoost, LightGBM, and CatBoost are optional dependencies; unavailable packages are explicitly reported and skipped.

In [ ]:
negative_count = int((y_train == 0).sum())
positive_count = int((y_train == 1).sum())
positive_weight = negative_count / positive_count

classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE
    ),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}

try:
    from xgboost import XGBClassifier
except ImportError:
    print("Skipping XGBoost; install with: pip install xgboost")
else:
    classifiers["XGBoost"] = XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=positive_weight, eval_metric="aucpr",
        n_jobs=-1, random_state=RANDOM_STATE,
    )

try:
    from lightgbm import LGBMClassifier
except ImportError:
    print("Skipping LightGBM; install with: pip install lightgbm")
else:
    classifiers["LightGBM"] = LGBMClassifier(
        n_estimators=300, learning_rate=0.05, class_weight="balanced",
        verbosity=-1, n_jobs=-1, random_state=RANDOM_STATE,
    )

try:
    from catboost import CatBoostClassifier
except ImportError:
    print("Skipping CatBoost; install with: pip install catboost")
else:
    classifiers["CatBoost"] = CatBoostClassifier(
        iterations=300, depth=6, learning_rate=0.05,
        auto_class_weights="Balanced", verbose=False,
        allow_writing_files=False, random_seed=RANDOM_STATE,
    )

print("Candidates:", list(classifiers))

## 7. Cross-validated model comparison

Average precision is the primary ranking metric because the positive class may be uncommon. ROC-AUC and balanced accuracy are secondary checks. Failures are raised rather than silently ignored.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "average_precision": "average_precision",
    "roc_auc": "roc_auc",
    "balanced_accuracy": "balanced_accuracy",
}

comparison_rows = []
candidate_pipelines = {}
for name, classifier in classifiers.items():
    pipeline = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("classifier", classifier),
    ])
    candidate_pipelines[name] = pipeline
    scores = cross_validate(
        pipeline, X_train, y_train, cv=cv, scoring=scoring,
        n_jobs=1, error_score="raise", return_train_score=False,
    )
    comparison_rows.append({
        "model": name,
        "average_precision": scores["test_average_precision"].mean(),
        "roc_auc": scores["test_roc_auc"].mean(),
        "balanced_accuracy": scores["test_balanced_accuracy"].mean(),
        "fit_time_seconds": scores["fit_time"].mean(),
    })

comparison = pd.DataFrame(comparison_rows).set_index("model").sort_values(
    "average_precision", ascending=False
)
display(comparison.round(4))
comparison.to_csv(REPORT_DIR / "model_comparison.csv")
best_name = comparison.index[0]
print("Top cross-validation candidate:", best_name)

## 8. Hyperparameter tuning

Tune only the top cross-validation candidate to keep runtime manageable. `RandomizedSearchCV` uses training data only and ranks settings by average precision. Increase `N_ITER` for a larger search budget.

In [ ]:
from scipy.stats import randint, loguniform, uniform

search_spaces = {
    "Logistic Regression": {
        "classifier__C": loguniform(1e-3, 1e2),
        "classifier__class_weight": [None, "balanced"],
    },
    "Random Forest": {
        "classifier__n_estimators": randint(200, 601),
        "classifier__max_depth": [None, 10, 20, 30],
        "classifier__min_samples_leaf": randint(1, 6),
        "classifier__max_features": ["sqrt", 0.7],
        "classifier__class_weight": [None, "balanced", "balanced_subsample"],
    },
    "Extra Trees": {
        "classifier__n_estimators": randint(200, 601),
        "classifier__max_depth": [None, 10, 20, 30],
        "classifier__min_samples_leaf": randint(1, 6),
        "classifier__max_features": ["sqrt", 0.7],
        "classifier__class_weight": [None, "balanced"],
    },
    "Gradient Boosting": {
        "classifier__n_estimators": randint(100, 401),
        "classifier__learning_rate": loguniform(0.01, 0.2),
        "classifier__max_depth": randint(1, 5),
        "classifier__min_samples_leaf": randint(1, 11),
    },
}

if best_name == "XGBoost":
    search_spaces[best_name] = {
        "classifier__n_estimators": randint(200, 601),
        "classifier__max_depth": randint(3, 9),
        "classifier__learning_rate": loguniform(0.01, 0.2),
        "classifier__subsample": uniform(0.7, 0.3),
        "classifier__colsample_bytree": uniform(0.7, 0.3),
    }
elif best_name == "LightGBM":
    search_spaces[best_name] = {
        "classifier__n_estimators": randint(200, 601),
        "classifier__num_leaves": randint(15, 64),
        "classifier__learning_rate": loguniform(0.01, 0.2),
        "classifier__min_child_samples": randint(10, 51),
        "classifier__subsample": uniform(0.7, 0.3),
    }
elif best_name == "CatBoost":
    search_spaces[best_name] = {
        "classifier__iterations": randint(200, 601),
        "classifier__depth": randint(4, 9),
        "classifier__learning_rate": loguniform(0.01, 0.2),
    }

N_ITER = 12
search = RandomizedSearchCV(
    estimator=candidate_pipelines[best_name],
    param_distributions=search_spaces[best_name],
    n_iter=N_ITER,
    scoring="average_precision",
    cv=cv,
    refit=True,
    random_state=RANDOM_STATE,
    n_jobs=1,
    error_score="raise",
    return_train_score=False,
)
search.fit(X_train, y_train)
best_model = search.best_estimator_
print("Best model:", best_name)
print("Best CV average precision:", round(search.best_score_, 4))
print("Best parameters:", search.best_params_)

## 9. Choose a decision threshold without using the test set

Use out-of-fold probabilities from the training partition to choose a threshold that meets a configurable minimum precision. If no threshold meets the constraint, use the threshold with the best F1 score. The final test partition remains untouched until the next section.

In [ ]:
oof_probability = cross_val_predict(
    clone(best_model), X_train, y_train, cv=cv,
    method="predict_proba", n_jobs=1,
)[:, 1]

MINIMUM_PRECISION = 0.70
precision, recall, thresholds = precision_recall_curve(y_train, oof_probability)
valid = precision[:-1] >= MINIMUM_PRECISION

if valid.any():
    valid_indices = np.flatnonzero(valid)
    chosen_index = valid_indices[np.argmax(recall[:-1][valid])]
    decision_threshold = float(thresholds[chosen_index])
    threshold_rule = f"highest recall with precision >= {MINIMUM_PRECISION:.2f}"
else:
    f1_values = 2 * precision[:-1] * recall[:-1] / np.maximum(
        precision[:-1] + recall[:-1], 1e-12
    )
    chosen_index = int(np.argmax(f1_values))
    decision_threshold = float(thresholds[chosen_index])
    threshold_rule = "best out-of-fold F1 (precision constraint not reached)"

print("Threshold:", round(decision_threshold, 4))
print("Selection rule:", threshold_rule)

## 10. Final held-out test evaluation

Fit the tuned pipeline on the training partition and evaluate once on the held-out test partition. The selected threshold was determined without test labels.

In [ ]:
best_model.fit(X_train, y_train)
y_probability = best_model.predict_proba(X_test)[:, 1]
y_prediction = (y_probability >= decision_threshold).astype(int)

metrics = {
    "selected_model": best_name,
    "decision_threshold": decision_threshold,
    "accuracy": accuracy_score(y_test, y_prediction),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_prediction),
    "precision_yes": precision_score(y_test, y_prediction, zero_division=0),
    "recall_yes": recall_score(y_test, y_prediction, zero_division=0),
    "f1_yes": f1_score(y_test, y_prediction, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_probability),
    "average_precision": average_precision_score(y_test, y_probability),
    "log_loss": log_loss(y_test, y_probability),
    "brier_score": brier_score_loss(y_test, y_probability),
}
display(pd.Series(metrics, name="test_metric").to_frame().round(4))
print(classification_report(
    y_test, y_prediction, labels=[0, 1], target_names=["no", "yes"],
    zero_division=0,
))
print("Confusion matrix (rows=actual, columns=predicted):")
print(confusion_matrix(y_test, y_prediction, labels=[0, 1]))

## 11. Evaluation plots and report files

In [ ]:
pd.DataFrame([metrics]).to_csv(REPORT_DIR / "test_metrics.csv", index=False)
report = classification_report(
    y_test, y_prediction, labels=[0, 1], target_names=["no", "yes"],
    output_dict=True, zero_division=0,
)
pd.DataFrame(report).transpose().to_csv(REPORT_DIR / "classification_report.csv")

cm = confusion_matrix(y_test, y_prediction, labels=[0, 1])
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["no", "yes"], yticklabels=["no", "yes"], ax=ax)
ax.set(xlabel="Predicted", ylabel="Actual", title="Confusion Matrix")
fig.tight_layout()
fig.savefig(REPORT_DIR / "confusion_matrix.png", dpi=150)
plt.show()
plt.close(fig)

fpr, tpr, _ = roc_curve(y_test, y_probability)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, label=f"ROC-AUC = {metrics['roc_auc']:.3f}")
ax.plot([0, 1], [0, 1], "k--", label="Chance")
ax.set(xlabel="False Positive Rate", ylabel="True Positive Rate", title="ROC Curve")
ax.legend()
fig.tight_layout()
fig.savefig(REPORT_DIR / "roc_curve.png", dpi=150)
plt.show()
plt.close(fig)

pr_precision, pr_recall, _ = precision_recall_curve(y_test, y_probability)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(pr_recall, pr_precision, label=f"Average precision = {metrics['average_precision']:.3f}")
ax.set(xlabel="Recall", ylabel="Precision", title="Precision-Recall Curve")
ax.legend()
fig.tight_layout()
fig.savefig(REPORT_DIR / "precision_recall_curve.png", dpi=150)
plt.show()
plt.close(fig)

print(f"Reports and plots saved in: {REPORT_DIR.resolve()}")

## 12. Save model and inference metadata

The saved estimator includes its preprocessing pipeline. Store the decision threshold and feature contract alongside it.

In [ ]:
MODEL_PATH = REPORT_DIR / "bank_subscription_pipeline.joblib"
METADATA_PATH = REPORT_DIR / "model_metadata.json"

joblib.dump(best_model, MODEL_PATH)
metadata = {
    "target": TARGET,
    "positive_label": "yes",
    "excluded_columns": excluded_columns,
    "feature_columns": X.columns.tolist(),
    "selected_model": best_name,
    "decision_threshold": decision_threshold,
    "threshold_selection_rule": threshold_rule,
    "random_state": RANDOM_STATE,
}
METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Saved model:", MODEL_PATH.resolve())
print("Saved metadata:", METADATA_PATH.resolve())

## 13. Example production inference

Input records must contain the same feature columns used for training. Validate incoming schemas and monitor prediction quality after deployment.

In [ ]:
loaded_model = joblib.load(MODEL_PATH)
saved_metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))

new_records = X_test.iloc[:3].copy()
expected_columns = saved_metadata["feature_columns"]
missing_columns = sorted(set(expected_columns) - set(new_records.columns))
extra_columns = sorted(set(new_records.columns) - set(expected_columns))
if missing_columns or extra_columns:
    raise ValueError(f"Input schema mismatch; missing={missing_columns}, extra={extra_columns}")
new_records = new_records[expected_columns]

subscription_probability = loaded_model.predict_proba(new_records)[:, 1]
subscription_prediction = (
    subscription_probability >= saved_metadata["decision_threshold"]
)
inference_result = pd.DataFrame({
    "probability_yes": subscription_probability,
    "predicted_label": np.where(subscription_prediction, "yes", "no"),
})
display(inference_result)

## Production checklist

- Confirm the prediction moment and exclude unavailable-at-inference fields (especially `duration` for pre-call predictions).
- Review target balance, class-specific precision/recall, average precision, ROC-AUC, and the confusion matrix; accuracy alone can mislead on imbalanced data.
- Select the operating threshold from validation or out-of-fold predictions, based on business costs; do not tune against the final test set.
- Add time-based validation if deployment predicts future campaigns, and check for duplicate customers or campaign leakage across splits.
- Pin package versions, test model loading in the serving environment, monitor input schema and drift, and retrain under a documented process.
- Benchmark optional boosting packages on representative data. More model families and larger searches do not guarantee better production performance.